## 논문 분석


### Claim Verification in the Age of Large Language Models: A Survey <br> - 2026.July
Alphaeus Dmonte, Roland Oruche, Marcos Zampieri, Prasad Calyam, Isabelle Augenstein <br>
(George Mason University, USA / University of Missouri-Columbia, USA / University of Copenhagen, Denmark)

### User-Centric Evidence Ranking for Attribution and Fact Verification <br> - 2026.Jan
Guy Alt, Eran Hirsch, Serwar Basch, Ido Dagan, Oren Glickman (Computer Science Department, Bar-Ilan University, Ramat Gan, Israel UKP Lab, TU Darmstadt, Germany)

---

### 프로젝트 목표와 Survey 논문의 연결

프로젝트에서 요구하는 전체 파이프라인은

```text
뉴스 기사
↓
검증 가능한 주장 탐지
↓
주장 유형 분류
↓
기간·단위·모집단 추출
↓
KOSIS 조회
↓
수치 비교
↓
판정
↓
설명 생성
```
입니다. 

반면 Survey 논문의 Claim Verification Pipeline은
```text
Claim Detection
↓
Check-worthiness
↓
Claim Matching
↓
Evidence Retrieval
↓
Evidence Selection
↓
Veracity Prediction
↓
Explanation Generation
```
입니다. 

두 구조를 비교하면 거의 1:1로 대응됩니다.

| Survey 논문              | 우리 프로젝트      |
| ---------------------- | ------------ |
| Claim Detection        | 검증 가능한 주장 탐지 |
| Check-worthiness       | 수치 기반 주장 선별  |
| Evidence Retrieval     | KOSIS API 검색 |
| Evidence Selection     | 공식 통계 선택     |
| Veracity Prediction    | 일치/불일치/판단불가  |
| Explanation Generation | LLM 설명 생성    |

즉, 프로젝트에서 요구하는 구조 자체가 최신 Claim Verification Pipeline과 거의 동일합니다.

---

### 우리 프로젝트에 실제 적용 가능한 기술

#### 1. Claim Detection

Survey에서 가장 먼저 수행하는 단계입니다. 우리 프로젝트에서는 기사 전체를 검증하는 것이 아니라 검증 가능한 문장만 추출해야 합니다.

예를 들어
```
한국 경제가 어렵다.
```

↓

검증 불가능

반면

```
청년 실업률은 7.2%이다.
```

↓

검증 가능

프로젝트 요구사항도 **수치 포함 문장 중심으로 검증 가능한 주장 탐지** 를 첫 단계로 요구하고 있습니다. 

---

#### 2. Claim Decomposition (추가 제안)

프로젝트 문서에서는 다음 정보를 추출하도록 요구합니다.
* 기간
* 단위
* 모집단



Survey에서는 최근 연구들이 복잡한 Claim을 Sub-Claim으로 나누는 것이 성능 향상에 도움이 된다고 설명합니다. 

예를 들어

```
2024년 국내 과수 농가의
65세 이상 비율은
64.2%이다.
```

를 다음처럼 분해할 수 있습니다.

```
연도

2024

↓

모집단

국내 과수 농가

↓

지표

65세 이상 비율

↓

수치

64.2%
```

이 구조는 이후 KOSIS API 검색에도 직접 활용할 수 있습니다.

---

#### 3. Retrieval → KOSIS API

Survey에서는 Evidence Retrieval을 가장 중요한 단계라고 설명합니다.

하지만 우리 프로젝트에서는 Evidence가 위키피디아나 뉴스가 아니라 **KOSIS 공식 통계**입니다.

즉,

```
Claim

↓

Retriever

↓

KOSIS API

↓

공식 통계
```

형태가 됩니다.

따라서 Survey의 Retrieval 부분을 KOSIS Retrieval로 바꾸면 됩니다.

---

#### 4. Retrieval 개선 아이디어

Survey에서는 단순 검색보다 다음 방법들이 효과적이라고 소개합니다.

* Query Optimization
* Re-ranking
* Multi-hop Retrieval



이 프로젝트에도 그대로 적용 가능합니다. 예를 들어, 기사에서

```
국내 과수 농가
```

라고 적혀 있어도

KOSIS에는

```
영농형태 : 과수
```

로 저장되어 있을 수 있습니다.

따라서, LLM을 이용해

```
기사 표현

↓

표준 검색어

↓

KOSIS 검색
```

으로 변환하면 검색 성공률이 높아질 수 있습니다.

---

### 5. 자동 계산 모듈

프로젝트 자료에서도 다음 계산을 수행합니다.

```
106877

/

166558

=

64.2%
```



즉, 우리 시스템은 단순 조회가 아니라 자동 계산이 필요합니다.

Survey에서는 Reasoning을 이용하여 계산 및 추론을 수행하는 연구들을 소개합니다. 

---

### 6. Explanation Generation

Survey에서 최근 가장 강조되는 부분입니다. 우리 프로젝트에서도 단순히

```
False
```

만 출력하지 않고

```
기사에서는

64.2%

라고 주장하였다.

↓

KOSIS API 조회 결과

64.17%

↓

소수 둘째 자리 반올림 시

64.2%

↓

일치
```

처럼 설명을 생성할 수 있습니다. 프로젝트 요구사항에도 설명 생성 기능이 포함되어 있습니다. 

---

### 프로젝트에 추가하면 좋은 연구 아이디어

Survey를 읽어보면 프로젝트에서 요구하지는 않지만 **차별화 요소**로 넣을 수 있는 기능들이 있습니다.

##### ① Query Rewriting

기사 표현을 KOSIS 검색어로 변환

```
기사

↓

LLM

↓

KOSIS 검색어 생성
```

---

##### ② Multi-hop Verification

현재 프로젝트는 한 개 통계표만 조회하는 수준입니다. 하지만,

```
청년 실업률이 증가했다.

↓

경제활동인구

+

실업자 수

↓

직접 계산
```

처럼 여러 통계를 조합하면 더 다양한 검증이 가능합니다.

---

##### ③ Explanation 개선

현재는 설명 생성 정도입니다. Survey에서는 Reasoning 과정까지 설명하도록 권장합니다. 예를 들어,

```
기사

↓

KOSIS 조회

↓

기간 비교

↓

모집단 비교

↓

단위 비교

↓

계산

↓

판정
```

과정을 그대로 보여줄 수 있습니다.

---

##### ④ Human Review

프로젝트 자료에서도 검증자 리뷰를 마지막 단계로 둡니다. Survey에서도 LLM은 환각(Hallucination), 편향(Bias), 오래된 지식 등으로 인해 완전한 자동 판정보다는 **의사결정 지원 도구**로 활용하는 것이 적절하다고 강조합니다. 

---

##### 이전에 읽은 Evidence Ranking 논문과의 연계

현재 프로젝트는 KOSIS를 주된 근거로 사용하기 때문에, 뉴스 기사 여러 개를 비교하는 일반적인 Fact Checking보다는 **공식 통계 기반 검증**에 초점을 맞추고 있습니다. 그래도 Evidence Ranking 논문의 아이디어는 다음과 같이 응용할 수 있습니다.

```
기사 주장

↓

KOSIS 검색

↓

후보 통계표 여러 개 발견

↓

가장 관련성이 높은 통계표부터 정렬

↓

LLM 검증
```

예를 들어 "고용"이라는 표현이 포함된 기사라면 KOSIS에서 여러 고용 관련 통계표가 검색될 수 있습니다. 이때 단순 키워드 일치가 아니라 **주장과 가장 잘 대응하는 통계표를 우선순위로 정렬**하면 잘못된 통계를 선택할 가능성을 줄일 수 있습니다. 즉, 이 프로젝트에서는 **문장(Evidence)을 랭킹하는 대신 통계표(Table)를 랭킹**하는 형태로 Evidence Ranking 개념을 확장해 볼 수 있습니다.

---

#### 발표에서 강조하면 좋은 차별점

이 Survey 논문를 단순히 "LLM을 사용한다"는 수준에서 인용하기보다, **프로젝트 설계의 이론적 근거**로 활용하는 것이 좋습니다.

* **Survey 논문**은 최신 LLM 기반 Claim Verification 파이프라인(Claim Detection → Retrieval → Veracity Prediction → Explanation)을 정리한 기준 모델을 제공합니다.
* **우리 프로젝트**는 이 파이프라인을 **공식 통계(KOSIS) 기반 수치 검증**에 특화하여 구현합니다.
* 추가적으로 **Evidence Ranking 논문의 아이디어를 통계표 랭킹(Table Ranking)** 으로 확장하면, 여러 KOSIS 통계표 중 가장 적합한 근거를 선택하는 차별화된 구조를 제안할 수 있습니다.

즉, 발표에서는 **"최신 LLM 기반 사실검증 파이프라인(Survey)을 기반으로 설계하고, 공식 통계 기반 검증과 통계표 랭킹을 결합한 뉴스 사실검증 시스템"**이라는 점을 강조하면 프로젝트의 연구적 근거와 차별성을 함께 보여줄 수 있습니다.
